# Project: (your title)
This notebook is the **control panel** of your project: it runs the whole project, step by step.

### ▶ Quick start – run the whole project
1. Open this file from Google Drive: `MyDrive/PyDS_project/main.ipynb` → *Open with → Google Colaboratory*.
2. **Runtime → Run all**. Click **Allow** when Google asks for access to Drive.
3. Every section below runs one step of the project and shows its result. That's all.

### How it works
- The logic lives in the modules in `src/` (`loader.py`, later `stats.py`, …). This notebook only **calls** them.
- You changed a module? Just **Run all** again – the first cell always loads the newest version of `src/`.
- After each lab you add one short section here, so at the end **Run all** = your whole analysis.

Folders: `data/raw` original data · `data/processed` data made by your code · `results/` tables and figures · `docs/` concept, data dictionary, block diagram.

In [8]:
# Setup – run this cell first every time you open the notebook
import sys
from pathlib import Path

PROJECT_NAME = "PyDS_project"          # the name of this project folder on Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_NAME
except ImportError:              # running locally (VS Code, Jupyter)
    PROJECT = Path.cwd()

for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]        # always use the newest version of src/
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:                    # not in Colab, or autoreload broken (Python 3.13) – src/ is reloaded anyway
    pass
print("Project:", PROJECT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/PyDS_project


## Lab 2 – Data and types (`src/loader.py`)

In [9]:
from src.loader import load_csv, convert_rows, get_column, DATA_FILE_NAME, COLUMN_TYPES

raw = load_csv(DATA_FILE_NAME)          # all values are text
print("rows:", len(raw), "| columns in the file:", list(raw[0]))

rows = convert_rows(raw, COLUMN_TYPES)  # only your columns, converted
print("first converted row:", rows[0])

rows: 649 | columns in the file: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']
first converted row: {'address': 'U', 'famsize': 'GT3', 'Medu': 4, 'Fedu': 4, 'studytime': 2, 'schoolsup': True, 'Walc': 1, 'absences': 4, 'G3': 11}


In [10]:
# Data overview – the tools from the Lab 2 exercises E6, E8, E9:
# 1. unique values of a text column  -> set(get_column(rows, "..."))
# 2. how often one value occurs      -> get_column(rows, "...").count("...")
# 3. min / max / mean of a number column

## Lab 3 – Functions and statistics (`src/stats.py`)

In [11]:
# Lab 3 – your code here

from src.loader import get_column
from src.stats import count_by, group_stat, mean, std

# 2. Cantidad de estudiantes por grupo de apoyo escolar
per_group = count_by(rows, "schoolsup")
print("rows per group:", per_group)

# 3. Nota media final (G3) según apoyo escolar
g3_per_group = group_stat(rows, "schoolsup", "G3")
print("mean G3 per group:", g3_per_group)

# 4. Desviación estándar de G3 usando get_column
g3_std = std(get_column(rows, "G3"))
print("G3 standard deviation:", g3_std)

rows per group: {True: 68, False: 581}
mean G3 per group: {True: 11.28, False: 11.98}
G3 standard deviation: 3.23


## Lab 4 – Classes (`src/models.py`)

In [12]:
# Lab 4 – your code here
from src.models import Dataset, CountBy, GroupMean, TopGroups, Equals, Above

# 1. Create the Dataset using 'rows' (loaded in the Lab 2 section)
students_data = Dataset(rows, "student grades")
print(students_data)

# 2. Define and run the analyses using your columns ("address" and "G3")
analyses = [
    CountBy("address"),
    GroupMean("address", "G3"),
    TopGroups("address", "G3", 3)
]

for a in analyses:
    print(a.report(students_data))

# 3. Apply a filter (passing students with final grade G3 > 10)
passing_students = Above("G3", 10).apply(students_data)
print(passing_students)

# 4. Run the analysis on the filtered subset
print(TopGroups("address", "G3", 3).report(passing_students))

# Finding: Urban students ("U") make up the majority of the dataset and achieve a slightly higher average final grade (G3) than rural students ("R"), a trend that holds even when filtering for passing students (G3 > 10).

Dataset 'student grades': 649 rows
count by address: {'U': 452, 'R': 197}
mean G3 by address: {'U': 12.26, 'R': 11.09}
top 3 address by mean G3: [('U', 12.26), ('R', 11.09)]
Dataset 'student grades | G3 > 10': 452 rows
top 3 address by mean G3: [('U', 13.53), ('R', 13.14)]


## Lab 5 – Errors and cleaning (`src/cleaning.py`)

In [14]:
# 1. Import clean, REQUIRED, and NON_NEGATIVE from src.cleaning
from src.cleaning import clean, REQUIRED, NON_NEGATIVE

# 2. Run clean on rows (from the Lab 2 section) with your two lists
clean_rows, report = clean(rows, REQUIRED, NON_NEGATIVE)

# 3. Print how many rows are left out of the original total
print(f"Kept {len(clean_rows)} of {len(rows)} rows")

# 4. Print the cleaning report
print("Cleaning report:", report)

# 5. Create a Dataset from clean_rows and print one of your analyses
clean_students = Dataset(clean_rows, "clean student grades")
print(GroupMean("address", "G3").report(clean_students))

# 6. Finding: No rows were removed (0 missing or negative values across all 649 rows), so the analysis results remain completely unchanged.

clean took 0.0008 s
Kept 649 of 649 rows
Cleaning report: {}
mean G3 by address: {'U': 12.26, 'R': 11.09}


## Lab 6 – Files and folders (`src/storage.py`)

In [ ]:
# Lab 6 – your code here

## Lab 7 – NumPy and plots (`src/viz.py`)

In [ ]:
# Lab 7 – your code here

## Lab 8 – Conclusions

Write your conclusions here: what does the analysis show, and what does it change?